# MuJoCo Soccer Multi-Agent DRL Training
Run this notebook in Google Colab to train the agents using a free GPU.

**Make sure to enable GPU:** `Runtime > Change runtime type > Hardware accelerator > GPU`

In [ ]:
!pip install -q dm_control shimmy[dm-control-multi-agent] pettingzoo torch stable-baselines3 supersuit tensorboard

In [ ]:
import os
import supersuit as ss
from stable_baselines3 import PPO
from shimmy import DmControlMultiAgentCompatibilityV0
from pettingzoo.utils.wrappers.base_parallel import BaseParallelWrapper
from google.colab import drive

# Mount Google Drive to save the model safely even if Colab disconnects
drive.mount('/content/drive')

# ---------------------------------------------------------
# ADVANCED REWARD SHAPING (To speed up learning)
# ---------------------------------------------------------
class SoccerRewardShapingWrapper(BaseParallelWrapper):
    def __init__(self, env):
        super().__init__(env)
        
    def __reduce__(self):
        return (SoccerRewardShapingWrapper, (self.env,))

    def step(self, actions):
        obs, rewards, terminations, truncations, infos = self.env.step(actions)
        for agent in self.possible_agents:
            if not terminations[agent] and not truncations[agent]:
                vel_to_ball = obs[agent]['stats_vel_to_ball'][0]
                vel_ball_to_goal = obs[agent]['stats_vel_ball_to_goal'][0]
                rewards[agent] += (0.001 * vel_to_ball) + (0.005 * vel_ball_to_goal)
        return obs, rewards, terminations, truncations, infos

# 1. Create the Environment
print("Initializing MuJoCo Soccer 2v2 Environment...")
env = DmControlMultiAgentCompatibilityV0(
    team_size=2,
    render_mode=None
)

# Fix for pickling the inner environment
env._ezpickle_args = ()
env._ezpickle_kwargs = {"team_size": 2, "render_mode": None}

# Apply the Reward Shaping
env = SoccerRewardShapingWrapper(env)

# 2. Wrap for Stable Baselines 3
env = ss.pettingzoo_env_to_vec_env_v1(env)
env = ss.concat_vec_envs_v1(env, num_vec_envs=1, num_cpus=1, base_class='stable_baselines3')

print("Environment Ready for Training!")

In [ ]:
# 3. Initialize and Train the Model
model = PPO(
    "MultiInputPolicy",
    env,
    verbose=1,
    learning_rate=3e-4,
    batch_size=256,
    tensorboard_log="./logs/"
)

# We will run for 1,000,000 timesteps now since the rewards help them learn faster
print("Starting training for 1,000,000 timesteps...")
model.learn(total_timesteps=1000000)

# 4. Save the Model directly to Google Drive!
model.save("/content/drive/MyDrive/ppo_soccer_2v2_colab")
print("Model safely saved to your Google Drive as ppo_soccer_2v2_colab.zip!")